In [1]:
mapa_parecer = {
    'veridico': 1,
    'verdadeiro': 1,
    'inveridico': 0,
    'inconclusivo': 2
}

mapa_categoria = {
    'descontextualizacao': 0,
    'fake news': 1,
    'nao se aplica': 2,
    'distorcao de midia': -1,
    'extremismo antidemocratico': 4,
    'discurso de odio': 5
}

In [ ]:
import pandas as pd
import unicodedata

result = pd.read_pickle(r"result\final_result_curadoria.pkl")

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('utf-8')
    return x

def limpar_id(x):
    x = str(x).strip()
    if x.endswith('.mp4'):
        x = x[:-4]
    try:
        return str(int(float(x))) 
    except:
        return x

def normalizar(texto):
    texto = texto.strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto

# Aplicar normalização nas colunas 'parecer' e 'categoria'
result['parecer'] = result['parecer'].apply(normalizar_texto)
result['categoria'] = result['categoria'].apply(normalizar_texto)

result.rename(columns={'id':'video_id'}, inplace=True)
result['video_id'] = result['video_id'].apply(limpar_id)

result['categoria'] = result['categoria'].apply(lambda x: mapa_categoria.get(normalizar(x), 2))
result['parecer'] = result['parecer'].apply(lambda x: mapa_parecer.get(normalizar(x), 2))
result

,video_id,link,veridico,inveridico,inconclusivas,categoria,parecer,confianca,justificativa
0,1,https://www.youtube.com/watch?v=tUsAxz2Df44,"""Adriana Acorsi do PT, apoiada pelo Lula. Esse...","""Saiu uma pesquisa recente feita nas capitais ...","""E aí, impressionante, de repente, todos aquel...",0,0,Confiante,O vídeo analisado conta com a exposição de um ...
1,10,https://www.youtube.com/watch?v=H_T5mCdOlrg,"""Qual foi a minha denúncia? O senhor é preside...","""Essa que é a verdade minha gente. Povo de Goi...","""Presidentes de Comissão é que pautam os assun...",0,2,Confiante,O conteúdo elaborado pelo autor do vídeo possu...
2,105,https://www.youtube.com/watch?v=_YR_-s5-xDM,"No vídeo seguinte, uma gravação feita de um ce...","""Mas porquê que o Danilo aqui está com a camis...","""Vai fazer o L pra vocês, agora, neste exato m...",1,0,Confiante,"No conteúdo analisado, a pessoa que elaborou o..."
3,11,https://www.youtube.com/watch?v=ywdoZSCGXDc,"""Aí vem gente fala: “Ah, você criticou ai, tav...","""Por que que querem prender o meu pai? Provave...","""Ah, Bolsonaro caiu dentro da prisão. Ah, poxa...",0,0,Confiante,O autor do vídeo faz uso de uma narrativa cont...
4,111,https://www.youtube.com/watch?v=HmYEpxVjajA,"""Sabe? Os negros não foram libertados pra vira...","""Eles deixaram de ser escravos pra virar vagab...","1% da população que come, que viaja, que vai p...",1,0,Muito confiante,O vídeo elaborado é feito de caso pensado e co...
...,...,...,...,...,...,...,...,...,...
114,201,NaN,NaN,NaN,NaN,5,0,muito confiante,Um suposto vídeo do presidente Lula mostra o p...
115,202,NaN,NaN,NaN,NaN,5,0,muito confiante,Um cidadão comenta uma notícia de que a advoga...
116,203,NaN,NaN,NaN,NaN,5,0,muito confiante,Um cidadão comenta em um vídeo sobre um supost...
117,204,NaN,NaN,NaN,NaN,2,1,muito confiante,Um jornal notícia a prisão do mecânico do 8 de...


In [10]:
df_real = result.copy()

In [ ]:
import os
import pandas as pd
import numpy as np
import unicodedata

# Caminho principal
base_path = r'results_llm_video_description_v3'

# Lista para armazenar os DataFrames
dataframes = []

# Percorre todas as subpastas
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith('.pkl'):
            file_path = os.path.join(root, file)
            model = os.path.basename(root)
            whisper_version = file

            # Lê o arquivo pkl
            try:
                df = pd.read_pickle(file_path)
                df['model'] = model
                df['whisper_version'] = whisper_version
                dataframes.append(df)
            except Exception as e:
                print(f"Erro ao ler {file_path}: {e}")

# Concatena todos os DataFrames
df_total_raw = pd.concat(dataframes, ignore_index=True)
df_total_raw = df_total_raw.loc[:, ~df_total_raw.columns.str.startswith('embedding_')]
df_total_raw = df_total_raw.loc[:, ~df_total_raw.columns.str.startswith('nome_base')]


df_total_raw = df_total_raw.dropna(subset=['label', 'categoria'])
df_total_raw['video_id'] = pd.to_numeric(df_total_raw['video_id'], errors='coerce').astype('Int64')
df_total_raw = df_total_raw.dropna(subset=['video_id'])
df_total_raw['video_id'] = df_total_raw['video_id'].astype(int)

df_total_raw.shape

(1428, 12)

In [12]:
df_total = df_total_raw.copy()

df_total['label'] = df_total['label'].apply(normalizar_texto)
df_total['categoria'] = df_total['categoria'].apply(normalizar_texto)

df_total['video_id'] = df_total['video_id'].apply(limpar_id)

df_total['categoria'] = df_total['categoria'].apply(lambda x: mapa_categoria.get(normalizar(x), 2))
df_total['label'] = df_total['label'].apply(lambda x: mapa_parecer.get(normalizar(x), 2))
df_total

,veridica,inveridica,inconclusivas,label,categoria,justification,video_id,tempo_processamento_seg,temperature_label,temperature_category,model,whisper_version
0,"<think>\nOkay, so I need to figure out the ver...","<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out how to ...",1,2,"<think>\nOkay, so I need to figure out the ver...",28,77.13,0.0,0.0,deepseek-r1-32b,large-v3_results.pkl
1,"<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out which s...",0,4,"<think>\nOkay, so I need to figure out how to ...",14,94.98,0.0,0.0,deepseek-r1-32b,large-v3_results.pkl
2,"<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out how to ...",0,5,"<think>\nOkay, so I need to help the user by p...",33,81.60,0.0,0.0,deepseek-r1-32b,large-v3_results.pkl
3,"<think>\nOkay, so I need to figure out how to ...","<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out how to ...",0,4,"<think>\nOkay, so I need to figure out how to ...",113,64.69,0.0,0.0,deepseek-r1-32b,large-v3_results.pkl
4,"<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out which s...","<think>\nOkay, so I need to figure out how to ...",0,0,"<think>\nOkay, so I need to figure out why the...",40,94.19,0.0,0.0,deepseek-r1-32b,large-v3_results.pkl
...,...,...,...,...,...,...,...,...,...,...,...,...
1661,- A imagem mostra uma pessoa sendo conduzida p...,- A pessoa está sendo conduzida por um policia...,- A pessoa na imagem está sendo conduzida por ...,0,2,Justificativa para o parecer e categoria finai...,198,5.39,0.0,0.0,qwen2.5-7b,large-v3_results.pkl
1662,- Pessoas em torno de uma mesa realizando uma ...,- Pessoas em torno de uma mesa realizando uma ...,- Pessoas em torno de uma mesa realizando uma ...,0,2,**Justificativa para o Parecer e Categoria Fin...,199,7.11,0.0,0.0,qwen2.5-7b,large-v3_results.pkl
1663,"- Diretório do PT sofre atos de vandalismo, de...",- Diretório do PT foi invadido e vandalizado e...,"- Diretório do PT de Adema foi depredado, vand...",0,0,Justificativa para o parecer final:\n\nA análi...,200,6.01,0.0,0.0,qwen2.5-7b,large-v3_results.pkl
1664,- A advogada presa pelo 8 de janeiro rompeu to...,- A advogada foi presa pelo 8 de janeiro.\n- E...,- A advogada presa pelo 8 de janeiro rompe tor...,0,4,**Justificativa para o Parecer e Categoria Fin...,202,8.76,0.0,0.0,qwen2.5-7b,large-v3_results.pkl


In [13]:
# Função para limpar IDs (tira espaços, converte para int se possível e depois para str)
def limpar_id(x):
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()

# Aplicar nos dois DataFrames
df_total['video_id'] = df_total['video_id'].apply(limpar_id)
df_real['video_id'] = df_real['video_id'].apply(limpar_id)

# Selecionar e renomear as colunas
result_filtrado = df_real[['video_id', 'categoria', 'parecer']].rename(
    columns={
        'categoria': 'categoria_real',
        'parecer': 'label_real'
    }
)

# Garantir tipos numéricos (int)
result_filtrado['categoria_real'] = result_filtrado['categoria_real'].astype(int)
result_filtrado['label_real'] = result_filtrado['label_real'].astype(int)

# Fazer o merge
df_total = df_total.merge(result_filtrado, on='video_id', how='left')
df_predict = df_total[['video_id', 'model', 'whisper_version', 'label', 'label_real', 'categoria', 'categoria_real']]

#Ajustando labels não normalizadas corretamente

df_predict.loc[(df_predict['video_id'] == '172') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0
df_predict.loc[(df_predict['video_id'] == '153') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0
df_predict.loc[(df_predict['video_id'] == '141') & (df_predict['model'] == 'phi3-14b'), 'label'] = 1
df_predict.loc[(df_predict['video_id'] == '134') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0
df_predict.loc[(df_predict['video_id'] == '1') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0

df_predict

,video_id,model,whisper_version,label,label_real,categoria,categoria_real
0,28,deepseek-r1-32b,large-v3_results.pkl,1,2,2,2
1,14,deepseek-r1-32b,large-v3_results.pkl,0,0,4,1
2,33,deepseek-r1-32b,large-v3_results.pkl,0,0,5,4
3,113,deepseek-r1-32b,large-v3_results.pkl,0,0,4,0
4,40,deepseek-r1-32b,large-v3_results.pkl,0,2,0,0
...,...,...,...,...,...,...,...
1423,198,qwen2.5-7b,large-v3_results.pkl,0,0,2,5
1424,199,qwen2.5-7b,large-v3_results.pkl,0,1,2,2
1425,200,qwen2.5-7b,large-v3_results.pkl,0,1,0,2
1426,202,qwen2.5-7b,large-v3_results.pkl,0,0,4,5


In [14]:
# Garante que não haja valores nulos nas colunas comparadas
df_final_temp_v2 = df_predict.dropna(subset=['label', 'label_real', 'categoria', 'categoria_real'])

# Colunas booleanas para acertos
df_final_temp_v2['acerto_label'] = (df_final_temp_v2['label'] == df_final_temp_v2['label_real']).astype(int)
df_final_temp_v2['acerto_categoria'] = (df_final_temp_v2['categoria'] == df_final_temp_v2['categoria_real']).astype(int)

# Agrupa por whisper_version e model
metricas = (
    df_final_temp_v2
    .groupby(['whisper_version', 'model'])
    .agg(
        total_exemplos=('video_id', 'count'),
        acertos_label=('acerto_label', 'sum'),
        acertos_categoria=('acerto_categoria', 'sum')
    )
    .reset_index()
)

# Calcula acurácias
metricas['acuracia_label'] = (metricas['acertos_label'] / metricas['total_exemplos']).round(3)
metricas['acuracia_categoria'] = (metricas['acertos_categoria'] / metricas['total_exemplos']).round(3)
metricas

,whisper_version,model,total_exemplos,acertos_label,acertos_categoria,acuracia_label,acuracia_categoria
0,large-v3_results.pkl,deepseek-r1-32b,119,75,49,0.630,0.412
1,large-v3_results.pkl,deepseek-r1-8b,119,64,37,0.538,0.311
2,large-v3_results.pkl,gemma2-9b,119,85,47,0.714,0.395
3,large-v3_results.pkl,gemma3-12b,119,76,66,0.639,0.555
4,large-v3_results.pkl,gpt-4.1-mini,119,76,62,0.639,0.521
5,large-v3_results.pkl,gpt-4o-mini,119,75,62,0.630,0.521
6,large-v3_results.pkl,llama3.1-8b,119,77,40,0.647,0.336
7,large-v3_results.pkl,llama3.2-3b,119,70,38,0.588,0.319
8,large-v3_results.pkl,phi3-14b,119,55,27,0.462,0.227
9,large-v3_results.pkl,qwen2-7b,119,66,39,0.555,0.328


In [15]:
import pandas as pd

# Remove casos com campos ausentes necessários
df_final_temp = df_predict.dropna(subset=['label', 'label_real', 'categoria', 'categoria_real'])

# Comparação com label_real como fonte de referência
df_real = df_final_temp.copy()
df_real['fonte_referencia'] = 'label_real'

# acurácia do label: ignora label_real == 2
df_real['acerto_label'] = None
df_real.loc[df_real['label_real'] != 2, 'acerto_label'] = (
    df_real.loc[df_real['label_real'] != 2, 'label'] == df_real.loc[df_real['label_real'] != 2, 'label_real']
).astype(int)

# acurácia da categoria: considera todas
df_real['acerto_categoria'] = (df_real['categoria'] == df_real['categoria_real']).astype(int)

# ---------- Parte 1: Métricas de label (com filtro em label_real != 2) ----------
df_metricas_label = (
    df_real[df_real['label_real'] != 2]  # somente exemplos válidos para acurácia de label
    .groupby(['fonte_referencia', 'whisper_version'])
    .agg(
        total_exemplos=('acerto_label', 'count'),
        acertos_label=('acerto_label', 'sum')
    )
    .reset_index()
)

# ---------- Parte 2: Métricas de categoria (com todos os exemplos) ----------
df_metricas_categoria = (
    df_real
    .groupby(['fonte_referencia', 'whisper_version'])
    .agg(
        total_exemplos_categoria=('acerto_categoria', 'count'),
        acertos_categoria=('acerto_categoria', 'sum')
    )
    .reset_index()
)

# ---------- Parte 3: Junta as métricas ----------
metricas = pd.merge(
    df_metricas_label,
    df_metricas_categoria,
    on=['fonte_referencia', 'whisper_version'],
    how='outer'
)

# ---------- Parte 4: Calcula acurácias ----------
metricas['acuracia_label'] = (metricas['acertos_label'] / metricas['total_exemplos']).round(3)
metricas['acuracia_categoria'] = (metricas['acertos_categoria'] / metricas['total_exemplos_categoria']).round(3)
metricas

,fonte_referencia,whisper_version,total_exemplos,acertos_label,total_exemplos_categoria,acertos_categoria,acuracia_label,acuracia_categoria
0,label_real,large-v3_results.pkl,1212,857,1428,570,0.707096,0.399


In [16]:
from sklearn.metrics import classification_report
import pandas as pd

relatorios = []

# Filtra valores válidos para garantir que não haja erro na comparação
df_filtrado = df_predict.dropna(subset=['label', 'label_real'])
df_filtrado = df_filtrado[~df_filtrado['video_id'].isin(['2', '127'])]# Remove os video_id indesejados

# Agrupa por modelo e versão do whisper
grupos = df_filtrado.groupby(['model', 'whisper_version'])

for (model, version), grupo in grupos:
    # Remove casos onde label_real == 2
    grupo = grupo[grupo['label_real'] != 2]

    # Garante que ainda há dados após o filtro
    if len(grupo) == 0:
        continue

    y_true = grupo['label_real'].astype(str)  # sklearn usa string por padrão
    y_pred = grupo['label'].astype(str)

    # Gera o relatório com métricas por classe
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    # Prepara base com modelo e versão
    resumo = {
        'model': model,
        'whisper_version': version,
        'accuracy': round(report['accuracy'], 3)
    }

    # Extrai métricas por classe (somente as chaves que são dígitos)
    for classe in sorted([k for k in report.keys() if k.isdigit()]):
        for metrica in ['precision', 'recall', 'f1-score']:
            chave = f'{metrica}_classe_{classe}'
            resumo[chave] = round(report[classe][metrica], 3)

    relatorios.append(resumo)

# Converte em DataFrame
df_metricas_por_classe = pd.DataFrame(relatorios)

# Calcula a média geral das métricas numéricas
media_geral = df_metricas_por_classe.drop(columns=['model', 'whisper_version']).mean().round(3)

# Cria a linha de média
linha_media = {
    'model': 'MÉDIA GERAL',
    'whisper_version': '-',
    **media_geral.to_dict()
}

# Adiciona ao DataFrame final
df_metricas_por_classe_com_media = pd.concat([
    df_metricas_por_classe,
    pd.DataFrame([linha_media])
], ignore_index=True)
df_metricas_por_classe_com_media.sort_values(
    by=['f1-score_classe_1', 'f1-score_classe_0'],
    ascending=[False, False]
).reset_index(drop=True)

,model,whisper_version,accuracy,precision_classe_0,recall_classe_0,f1-score_classe_0,precision_classe_1,recall_classe_1,f1-score_classe_1
0,gemma2-9b,large-v3_results.pkl,0.830,0.810,0.986,0.889,0.938,0.484,0.638
1,gpt-4.1-mini,large-v3_results.pkl,0.750,0.833,0.797,0.815,0.588,0.645,0.615
2,qwen2-7b,large-v3_results.pkl,0.650,0.925,0.536,0.679,0.467,0.903,0.615
3,llama3.1-8b,large-v3_results.pkl,0.760,0.785,0.899,0.838,0.667,0.452,0.538
4,phi3-14b,large-v3_results.pkl,0.540,0.848,0.406,0.549,0.388,0.839,0.531
5,deepseek-r1-8b,large-v3_results.pkl,0.630,0.786,0.638,0.704,0.432,0.613,0.507
6,deepseek-r1-32b,large-v3_results.pkl,0.740,0.753,0.928,0.831,0.667,0.323,0.435
7,MÉDIA GERAL,-,0.711,0.778,0.844,0.789,0.642,0.414,0.419
8,gemma3-12b,large-v3_results.pkl,0.740,0.736,0.971,0.838,0.778,0.226,0.350
9,qwen2.5-14b,large-v3_results.pkl,0.740,0.736,0.971,0.838,0.778,0.226,0.350


In [17]:
from sklearn.metrics import classification_report
import pandas as pd

relatorios = []

# Filtra valores válidos para garantir que não haja erro na comparação
df_filtrado = df_predict.dropna(subset=['categoria', 'categoria_real'])
df_filtrado = df_filtrado[~df_filtrado['video_id'].isin(['2', '127'])]# Remove os video_id indesejados

# Agrupa por modelo e versão do whisper
grupos = df_filtrado.groupby(['model', 'whisper_version'])

for (model, version), grupo in grupos:
    # Remove casos onde categoria_real == 2
    grupo = grupo[grupo['categoria_real'] != 2]

    # Garante que ainda há dados após o filtro
    if len(grupo) == 0:
        continue

    y_true = grupo['categoria_real'].astype(str)  # sklearn usa string por padrão
    y_pred = grupo['categoria'].astype(str)

    # Gera o relatório com métricas por classe
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    # Prepara base com modelo e versão
    resumo = {
        'model': model,
        'whisper_version': version,
        'accuracy': round(report['accuracy'], 3)
    }

    # Extrai métricas por classe (somente as chaves que são dígitos)
    for classe in sorted([k for k in report.keys() if k.isdigit()]):
        for metrica in ['precision', 'recall', 'f1-score']:
            chave = f'{metrica}_classe_{classe}'
            resumo[chave] = round(report[classe][metrica], 3)

    relatorios.append(resumo)

# Converte em DataFrame
df_metricas_por_classe = pd.DataFrame(relatorios)

# Calcula a média geral das métricas numéricas
media_geral = df_metricas_por_classe.drop(columns=['model', 'whisper_version']).mean().round(3)

# Cria a linha de média
linha_media = {
    'model': 'MÉDIA GERAL',
    'whisper_version': '-',
    **media_geral.to_dict()
}

# Adiciona ao DataFrame final
df_metricas_por_classe_com_media = pd.concat([
    df_metricas_por_classe,
    pd.DataFrame([linha_media])
], ignore_index=True)
df_metricas_por_classe_com_media.sort_values(
    by=['f1-score_classe_1', 'f1-score_classe_0'],
    ascending=[False, False]
).reset_index(drop=True)

,model,whisper_version,accuracy,precision_classe_0,recall_classe_0,f1-score_classe_0,precision_classe_1,recall_classe_1,f1-score_classe_1,precision_classe_2,recall_classe_2,f1-score_classe_2,precision_classe_4,recall_classe_4,f1-score_classe_4,precision_classe_5,recall_classe_5,f1-score_classe_5
0,gemma3-12b,large-v3_results.pkl,0.465,0.731,0.543,0.623,1.000,0.300,0.462,0.0,0.0,0.0,0.444,0.727,0.552,0.583,0.350,0.438
1,phi3-14b,large-v3_results.pkl,0.221,1.000,0.029,0.056,0.234,0.900,0.371,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000,0.000
2,llama3.2-3b,large-v3_results.pkl,0.314,0.500,0.429,0.462,0.389,0.350,0.368,0.0,0.0,0.0,0.133,0.182,0.154,0.231,0.150,0.182
3,gpt-4.1-mini,large-v3_results.pkl,0.407,0.514,0.543,0.528,0.556,0.250,0.345,0.0,0.0,0.0,0.500,0.364,0.421,0.778,0.350,0.483
4,qwen2.5-7b,large-v3_results.pkl,0.279,0.520,0.371,0.433,0.257,0.450,0.327,0.0,0.0,0.0,0.000,0.000,0.000,0.667,0.100,0.174
5,qwen2.5-14b,large-v3_results.pkl,0.465,0.565,0.743,0.642,1.000,0.150,0.261,0.0,0.0,0.0,0.389,0.636,0.483,0.667,0.200,0.308
6,MÉDIA GERAL,-,0.354,0.561,0.486,0.460,0.532,0.242,0.245,0.0,0.0,0.0,0.215,0.273,0.235,0.516,0.279,0.349
7,deepseek-r1-8b,large-v3_results.pkl,0.174,0.400,0.171,0.240,0.500,0.150,0.231,0.0,0.0,0.0,0.000,0.000,0.000,0.600,0.300,0.400
8,gpt-4o-mini,large-v3_results.pkl,0.488,0.532,0.714,0.610,0.667,0.100,0.174,0.0,0.0,0.0,0.429,0.545,0.480,0.692,0.450,0.545
9,qwen2-7b,large-v3_results.pkl,0.337,0.412,0.600,0.488,0.250,0.100,0.143,0.0,0.0,0.0,0.000,0.000,0.000,0.316,0.300,0.308


# Metrics to Paper

In [ ]:
from ollama import Client

api_host = ""
client = Client(host=api_host)

models = client.list()

print(r"\textbf{Version} & \textbf{Parameters} & \textbf{Quantization} & \textbf{Size (GB)} \\ \hline")

for model in models['models']:
    full_model = model.get('model', '')
    version = full_model.split(':')[0]
    parameters = model.get('details', {}).get('parameter_size', '—')
    quant = model.get('details', {}).get('quantization_level', '—')
    size_gb = round(model.get('size', 0) / (1024 ** 3), 2)

    print(f"{version} & {parameters} & {quant} & {size_gb} \\\\")


\textbf{Version} & \textbf{Parameters} & \textbf{Quantization} & \textbf{Size (GB)} \\ \hline
nomic-embed-text & 137M & F16 & 0.26 \\
llama3.2-vision & 10.7B & Q4_K_M & 7.28 \\
qwen2.5vl & 8.3B & Q4_K_M & 5.56 \\
gemma3 & 12.2B & Q4_K_M & 7.59 \\
deepseek-r1 & 8.0B & Q4_K_M & 4.58 \\
llama3.1 & 8.0B & Q4_K_M & 4.58 \\
llama3.2 & 3.2B & Q4_K_M & 1.88 \\
phi3 & 14.0B & Q4_0 & 7.35 \\
qwen2.5 & 14.8B & Q4_K_M & 8.37 \\
qwen2.5 & 7.6B & Q4_K_M & 4.36 \\
deepseek-r1 & 32.8B & Q4_K_M & 18.49 \\
gemma2 & 9.2B & Q4_0 & 5.07 \\
qwen2 & 7.6B & Q4_0 & 4.13 \\


In [23]:
import openai

# Carrega a chave da API a partir do arquivo
with open("key_openai.txt", "r") as f:
    api_key = f.read().strip()

# Configura o cliente OpenAI
client = openai.OpenAI(api_key=api_key)

# Lista de modelos para inspecionar
gpt_models = ['gpt-4.1-mini', 'gpt-4o-mini']

# Cabeçalho da tabela
print(r"\textbf{Developer} & \textbf{Version} & \textbf{Parameters} & \textbf{Quantization} & \textbf{Size (GB)} \\ \hline")

for model_name in gpt_models:
    try:
        # Pega as informações do modelo
        model_info = client.models.retrieve(model_name)

        # Estimativas manuais (OpenAI não fornece todos os dados via API publicamente)
        developer = "OpenAI"
        version = model_info.id
        parameters = "—"  # Não divulgado
        quantization = "—"  # Não divulgado
        size_gb = "—"  # Não divulgado

        print(f"{developer} & {version} & {parameters} & {quantization} & {size_gb} \\\\")
    
    except Exception as e:
        print(f"% Erro ao buscar modelo {model_name}: {e}")


\textbf{Developer} & \textbf{Version} & \textbf{Parameters} & \textbf{Quantization} & \textbf{Size (GB)} \\ \hline
OpenAI & gpt-4.1-mini & — & — & — \\
OpenAI & gpt-4o-mini & — & — & — \\


## Audio-based Results

In [ ]:
import pandas as pd
import unicodedata

result = pd.read_pickle(r"result\final_result_curadoria.pkl")

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('utf-8')
    return x

def limpar_id(x):
    x = str(x).strip()
    if x.endswith('.mp4'):
        x = x[:-4]
    try:
        return str(int(float(x))) 
    except:
        return x

def normalizar(texto):
    texto = texto.strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto

# Aplicar normalização nas colunas 'parecer' e 'categoria'
result['parecer'] = result['parecer'].apply(normalizar_texto)
result['categoria'] = result['categoria'].apply(normalizar_texto)

result.rename(columns={'id':'video_id'}, inplace=True)
result['video_id'] = result['video_id'].apply(limpar_id)

result['categoria'] = result['categoria'].apply(lambda x: mapa_categoria.get(normalizar(x), 2))
result['parecer'] = result['parecer'].apply(lambda x: mapa_parecer.get(normalizar(x), 2))
result

,video_id,link,veridico,inveridico,inconclusivas,categoria,parecer,confianca,justificativa
0,1,https://www.youtube.com/watch?v=tUsAxz2Df44,"""Adriana Acorsi do PT, apoiada pelo Lula. Esse...","""Saiu uma pesquisa recente feita nas capitais ...","""E aí, impressionante, de repente, todos aquel...",0,0,Confiante,O vídeo analisado conta com a exposição de um ...
1,10,https://www.youtube.com/watch?v=H_T5mCdOlrg,"""Qual foi a minha denúncia? O senhor é preside...","""Essa que é a verdade minha gente. Povo de Goi...","""Presidentes de Comissão é que pautam os assun...",0,2,Confiante,O conteúdo elaborado pelo autor do vídeo possu...
2,105,https://www.youtube.com/watch?v=_YR_-s5-xDM,"No vídeo seguinte, uma gravação feita de um ce...","""Mas porquê que o Danilo aqui está com a camis...","""Vai fazer o L pra vocês, agora, neste exato m...",1,0,Confiante,"No conteúdo analisado, a pessoa que elaborou o..."
3,11,https://www.youtube.com/watch?v=ywdoZSCGXDc,"""Aí vem gente fala: “Ah, você criticou ai, tav...","""Por que que querem prender o meu pai? Provave...","""Ah, Bolsonaro caiu dentro da prisão. Ah, poxa...",0,0,Confiante,O autor do vídeo faz uso de uma narrativa cont...
4,111,https://www.youtube.com/watch?v=HmYEpxVjajA,"""Sabe? Os negros não foram libertados pra vira...","""Eles deixaram de ser escravos pra virar vagab...","1% da população que come, que viaja, que vai p...",1,0,Muito confiante,O vídeo elaborado é feito de caso pensado e co...
...,...,...,...,...,...,...,...,...,...
114,201,NaN,NaN,NaN,NaN,5,0,muito confiante,Um suposto vídeo do presidente Lula mostra o p...
115,202,NaN,NaN,NaN,NaN,5,0,muito confiante,Um cidadão comenta uma notícia de que a advoga...
116,203,NaN,NaN,NaN,NaN,5,0,muito confiante,Um cidadão comenta em um vídeo sobre um supost...
117,204,NaN,NaN,NaN,NaN,2,1,muito confiante,Um jornal notícia a prisão do mecânico do 8 de...


In [228]:
df_real = result.copy()

In [ ]:
import os
import pandas as pd
import numpy as np
import unicodedata

# Caminho principal
base_path = r'results_llm_v3'

# Lista para armazenar os DataFrames
dataframes = []

# Percorre todas as subpastas
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith('.pkl'):
            file_path = os.path.join(root, file)
            model = os.path.basename(root)
            whisper_version = file

            # Lê o arquivo pkl
            try:
                df = pd.read_pickle(file_path)
                df['model'] = model
                df['whisper_version'] = whisper_version
                dataframes.append(df)
            except Exception as e:
                print(f"Erro ao ler {file_path}: {e}")

# Concatena todos os DataFrames
df_total_raw = pd.concat(dataframes, ignore_index=True)
df_total_raw = df_total_raw.loc[:, ~df_total_raw.columns.str.startswith('embedding_')]
df_total_raw = df_total_raw.loc[:, ~df_total_raw.columns.str.startswith('nome_base')]


df_total_raw = df_total_raw.dropna(subset=['label', 'categoria'])
df_total_raw['video_id'] = pd.to_numeric(df_total_raw['video_id'], errors='coerce').astype('Int64')
df_total_raw = df_total_raw.dropna(subset=['video_id'])
df_total_raw['video_id'] = df_total_raw['video_id'].astype(int)

df_total = df_total_raw.copy()

df_total['label'] = df_total['label'].apply(normalizar_texto)
df_total['categoria'] = df_total['categoria'].apply(normalizar_texto)

df_total['video_id'] = df_total['video_id'].apply(limpar_id)

df_total['categoria'] = df_total['categoria'].apply(lambda x: mapa_categoria.get(normalizar(x), 2))
df_total['label'] = df_total['label'].apply(lambda x: mapa_parecer.get(normalizar(x), 2))

def limpar_id(x):
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()

# Aplicar nos dois DataFrames
df_total['video_id'] = df_total['video_id'].apply(limpar_id)
df_real['video_id'] = df_real['video_id'].apply(limpar_id)

# Selecionar e renomear as colunas
result_filtrado = df_real[['video_id', 'categoria', 'parecer']].rename(
    columns={
        'categoria': 'categoria_real',
        'parecer': 'label_real'
    }
)

# Garantir tipos numéricos (int)
result_filtrado['categoria_real'] = result_filtrado['categoria_real'].astype(int)
result_filtrado['label_real'] = result_filtrado['label_real'].astype(int)

# Fazer o merge
df_total = df_total.merge(result_filtrado, on='video_id', how='left')
df_predict = df_total[['video_id', 'model', 'whisper_version', 'label', 'label_real', 'categoria', 'categoria_real']]

In [242]:
from sklearn.metrics import classification_report, f1_score
import pandas as pd

relatorios = []

# Filtrar valores válidos
df_filtrado = df_predict.dropna(subset=['label', 'label_real'])
df_filtrado = df_filtrado[
    (df_filtrado['label_real'] != 2) &
    (~df_filtrado['video_id'].isin(['2', '127'])) &
    (df_filtrado['whisper_version'] == 'large-v3_results.pkl')
]

# Agrupar por modelo
for model, grupo in df_filtrado.groupby('model'):
    y_true = grupo['label_real']
    y_pred = grupo['label']

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    resumo = {
        'model': model
    }

    # Adiciona métricas por label (0 e 1)
    for label in ['0', '1']:
        if label in report:
            resumo[f'f1_label_{label}'] = round(report[label]['f1-score'], 3)
        else:
            resumo[f'f1_label_{label}'] = 0.0

    # Adiciona F1 macro total
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    resumo['f1_label_macro'] = round(f1_macro, 3)

    relatorios.append(resumo)

# Criação do DataFrame final
df_metricas_por_label = pd.DataFrame(relatorios)
df_metricas_por_label = df_metricas_por_label.sort_values(by='f1_label_macro', ascending=False)
print(df_metricas_por_label)

            model  f1_label_0  f1_label_1  f1_label_macro
4     llama3.1-8b       0.859       0.324           0.394
2    gpt-4.1-mini       0.554       0.293           0.282
0  deepseek-r1-8b       0.452       0.381           0.278
5     llama3.2-3b       0.816       0.000           0.272
6        phi3-14b       0.375       0.381           0.252
8     qwen2.5-14b       0.447       0.244           0.230
9      qwen2.5-7b       0.458       0.216           0.225
1       gemma2-9b       0.395       0.263           0.220
7        qwen2-7b       0.029       0.486           0.171
3     gpt-4o-mini       0.135       0.121           0.085


In [245]:
from sklearn.metrics import classification_report, f1_score
import pandas as pd

relatorios = []

# Filtrar valores válidos
df_filtrado = df_predict.dropna(subset=['categoria', 'categoria_real'])
df_filtrado = df_filtrado[
    # (df_filtrado['label_real'] != 2) &
    # (~df_filtrado['video_id'].isin(['2', '127'])) &
    (df_filtrado['whisper_version'] == 'large-v3_results.pkl')
]
# df_filtrado = df_filtrado[~df_filtrado['video_id'].isin(['2', '127'])]

# Agrupar por modelo
for model, grupo in df_filtrado.groupby('model'):
    y_true = grupo['categoria_real']
    y_pred = grupo['categoria']

    # Gera o relatório completo
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    resumo = {'model': model}

    # Adiciona f1-score por classe individual
    for categoria in sorted(df_filtrado['categoria_real'].unique()):
        cat_str = str(categoria)
        # if cat_str in report:
        #     resumo[f'f1_categoria_{cat_str}'] = round(report[cat_str]['f1-score'], 3)
        # else:
        #     resumo[f'f1_categoria_{cat_str}'] = 0.0

    # F1 macro total
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    resumo['f1_categoria_macro'] = round(f1_macro, 3)

    relatorios.append(resumo)

# Criar DataFrame e ordenar pelo f1_macro
df_metricas_por_categoria = pd.DataFrame(relatorios)
df_metricas_por_categoria = df_metricas_por_categoria.sort_values(by='f1_categoria_macro', ascending=False)

# Exibir resultado
print(df_metricas_por_categoria)

            model  f1_categoria_macro
6        phi3-14b               0.450
2    gpt-4.1-mini               0.442
3     gpt-4o-mini               0.338
8     qwen2.5-14b               0.304
9      qwen2.5-7b               0.284
0  deepseek-r1-8b               0.275
1       gemma2-9b               0.260
7        qwen2-7b               0.250
4     llama3.1-8b               0.241
5     llama3.2-3b               0.184


In [180]:
import pandas as pd

# Suponha que df_predict já esteja carregado com as colunas:
# ['video_id', 'whisper_version', 'label', 'label_real']

# 1. Filtrar amostras com label diferente de 2
df_filtrado = df_predict[df_predict['label'] != 2]

# 2. Contar a quantidade de cada label por whisper_version
df_label_dist = df_filtrado.groupby(['whisper_version', 'label']).size().unstack(fill_value=0)

# 3. Calcular porcentagem por whisper_version
df_label_percent = df_label_dist.div(df_label_dist.sum(axis=1), axis=0).round(3)

# 4. Calcular a distribuição real (label_real) considerando apenas um rótulo por vídeo
df_real = df_predict[df_predict['label_real'] != 2].drop_duplicates('video_id')
df_real_dist = df_real.groupby('label_real').size().div(df_real['video_id'].nunique()).round(3)

# 5. Inserir linha com distribuição real na tabela de comparação
df_comparacao = df_label_percent.copy()
df_comparacao.loc['REAL'] = [df_real_dist.get(0, 0), df_real_dist.get(1, 0)]

# 6. Exibir resultado
print(df_comparacao)

label                     0      1
whisper_version                   
large-v3_results.pkl  0.701  0.299
large_results.pkl     0.713  0.287
tiny_results.pkl      0.843  0.157
REAL                  0.693  0.307


In [181]:
import pandas as pd

df_filtrado = df_predict.copy()

# 1. Contar quantidade de cada categoria por whisper_version
df_categoria_dist = df_filtrado.groupby(['whisper_version', 'categoria']).size().unstack(fill_value=0)

# 2. Calcular porcentagem por whisper_version
df_categoria_percent = df_categoria_dist.div(df_categoria_dist.sum(axis=1), axis=0).round(3)

# 3. Calcular a distribuição real por video_id único
df_real = df_predict.drop_duplicates('video_id')
df_real_dist = df_real['categoria_real'].value_counts(normalize=True).round(3)

# 4. Reindexar a linha real com as mesmas colunas do df_categoria_percent
real_row = df_categoria_percent.columns.to_series().apply(lambda x: df_real_dist.get(x, 0.0))

# 5. Inserir linha 'REAL' na tabela
df_comparacao = df_categoria_percent.copy()
df_comparacao.loc['REAL'] = real_row

# 6. Exibir resultado
print(df_comparacao)

categoria                 0      1      2      4      5
whisper_version                                        
large-v3_results.pkl  0.402  0.033  0.322  0.060  0.182
large_results.pkl     0.394  0.038  0.317  0.067  0.184
tiny_results.pkl      0.407  0.049  0.323  0.056  0.165
REAL                  0.294  0.168  0.269  0.101  0.168


In [186]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# Filtra rótulos válidos
df_label = df_predict[df_predict['label_real'] != 2]
df_label = df_label[~df_label['video_id'].isin(['2', '127'])]

# Identifica todas as classes únicas de label
labels_unicos = sorted(df_label['label_real'].unique())

# Lista para armazenar os resultados
metrics_por_label = []

# Agrupa por versão do whisper
for whisper_version, group in df_label.groupby('whisper_version'):
    y_true = group['label_real']
    y_pred = group['label']
    
    for label in labels_unicos:
        # Máscara para a classe atual
        y_true_bin = (y_true == label).astype(int)
        y_pred_bin = (y_pred == label).astype(int)

        precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)
        recall = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)

        metrics_por_label.append({
            'whisper_version': whisper_version,
            'label': label,
            'precision': round(precision, 3),
            'recall': round(recall, 3),
            'f1_score': round(f1, 3)
        })

# Criação do DataFrame final
df_metrics_label_detalhado = pd.DataFrame(metrics_por_label)
print(df_metrics_label_detalhado)

        whisper_version  label  precision  recall  f1_score
0  large-v3_results.pkl      0      0.802   0.387     0.522
1  large-v3_results.pkl      1      0.504   0.210     0.296
2     large_results.pkl      0      0.799   0.385     0.520
3     large_results.pkl      1      0.558   0.221     0.317
4      tiny_results.pkl      0      0.747   0.343     0.470
5      tiny_results.pkl      1      0.449   0.077     0.131


In [88]:
from sklearn.metrics import precision_score, f1_score
import pandas as pd

# Filtra rótulos válidos
# df_categoria = df_predict[df_predict['categoria_real'] != 2]
df_categoria = df_predict[~df_predict['video_id'].isin(['2', '127'])]

# Lista para armazenar as métricas
metrics_categoria = []

# Identifica todas as classes possíveis (categorias)
categorias_unicos = sorted(df_categoria['categoria_real'].unique())

# Agrupa por whisper_version
for whisper_version, group in df_categoria.groupby('whisper_version'):
    y_true = group['categoria_real']
    y_pred = group['categoria']

    # F1 macro e weighted
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Precisão por classe individual
    precision_por_classe = precision_score(y_true, y_pred, average=None, labels=categorias_unicos, zero_division=0)

    # Estrutura do dicionário
    entry = {
        'whisper_version': whisper_version,
        'f1_categoria_macro': round(f1_macro, 3),
        'f1_categoria_weighted': round(f1_weighted, 3)
    }

    # Adiciona a precisão por classe ao dicionário
    for categoria_val, prec in zip(categorias_unicos, precision_por_classe):
        entry[f'precision_categoria_{categoria_val}'] = round(prec, 3)

    metrics_categoria.append(entry)

# Criação do DataFrame final
df_metrics_categoria = pd.DataFrame(metrics_categoria)
print(df_metrics_categoria)


        whisper_version  f1_categoria_macro  f1_categoria_weighted  \
0  large-v3_results.pkl               0.324                  0.363   
1     large_results.pkl               0.317                  0.356   
2      tiny_results.pkl               0.288                  0.334   

   precision_categoria_0  precision_categoria_1  precision_categoria_2  \
0                  0.410                  0.222                  0.425   
1                  0.400                  0.300                  0.419   
2                  0.399                  0.327                  0.381   

   precision_categoria_4  precision_categoria_5  
0                  0.303                  0.357  
1                  0.250                  0.342  
2                  0.183                  0.284  


In [79]:
# Remove valores inválidos se necessário (por exemplo, NaNs)
df_cat = df_predict.dropna(subset=['categoria', 'categoria_real'])
# df_label = df_label[~df_label['video_id'].isin(['2', '127'])]

metrics_categoria = []

# Agrupa por whisper_version
for whisper_version, group in df_cat.groupby('whisper_version'):
    y_true = group['categoria_real']
    y_pred = group['categoria']

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

    metrics_categoria.append({
        'whisper_version': whisper_version,
        'accuracy_categoria': acc,
        'precision_categoria': prec,
        'recall_categoria': recall,
        'f1_categoria': f1,
        'f1_categoria_macro': f1_macro
    })

# DataFrame com as métricas finais
df_metrics_categoria = pd.DataFrame(metrics_categoria).round(3)
print(df_metrics_categoria)


        whisper_version  accuracy_categoria  precision_categoria  \
0  large-v3_results.pkl               0.388                0.359   
1     large_results.pkl               0.377                0.359   
2      tiny_results.pkl               0.355                0.339   

   recall_categoria  f1_categoria  f1_categoria_macro  
0             0.388         0.359               0.320  
1             0.377         0.351               0.314  
2             0.355         0.331               0.287  


In [80]:
# df_model_metrics contém colunas:
# ['model', 'whisper_version', 'f1_label', 'f1_categoria', ...]

# 1. Calcula o desvio padrão da F1-score para LABEL (macro ou weighted) entre os modelos por whisper_version
std_f1_label = df_model_metrics.groupby('whisper_version')['f1_label'].std().round(3)

# 2. Calcula o desvio padrão da F1-score para CATEGORIA
std_f1_categoria = df_model_metrics.groupby('whisper_version')['f1_categoria'].std().round(3)

# 3. Juntar ambos em um DataFrame para visualização
df_std_f1 = pd.DataFrame({
    'std_f1_label': std_f1_label,
    'std_f1_categoria': std_f1_categoria
})

print(df_std_f1)

                      std_f1_label  std_f1_categoria
whisper_version                                     
large-v3_results.pkl         0.165             0.080
large_results.pkl            0.171             0.055
tiny_results.pkl             0.156             0.065


In [70]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

metrics = []

# Remove os video_ids indesejados antes do agrupamento
df_filtrado = df_predict[~df_predict['video_id'].isin(['2', '127'])]

# Agrupa por model e whisper_version
for (model, whisper_version), df_group in df_filtrado.groupby(['model', 'whisper_version']):
    # Remove amostras onde label_real == 2
    df_group = df_group[df_group['label_real'] != 2]

    # Verificação extra: grupo pode ficar vazio
    if df_group.empty:
        continue

    # Métricas para "label" (weighted)
    acc_label = accuracy_score(df_group['label_real'], df_group['label'])
    prec_label = precision_score(df_group['label_real'], df_group['label'], average='weighted', zero_division=0)
    recall_label = recall_score(df_group['label_real'], df_group['label'], average='weighted', zero_division=0)
    f1_label = f1_score(df_group['label_real'], df_group['label'], average='weighted', zero_division=0)

    # Métricas para "categoria" (weighted)
    acc_cat = accuracy_score(df_group['categoria_real'], df_group['categoria'])
    prec_cat = precision_score(df_group['categoria_real'], df_group['categoria'], average='weighted', zero_division=0)
    recall_cat = recall_score(df_group['categoria_real'], df_group['categoria'], average='weighted', zero_division=0)
    f1_cat = f1_score(df_group['categoria_real'], df_group['categoria'], average='weighted', zero_division=0)

    metrics.append({
        'whisper_version': whisper_version,
        'model': model,
        'accuracy_label': acc_label,
        'precision_label': prec_label,
        'recall_label': recall_label,
        'f1_label': f1_label,
        'accuracy_categoria': acc_cat,
        'precision_categoria': prec_cat,
        'recall_categoria': recall_cat,
        'f1_categoria': f1_cat
    })

# Criar DataFrame final e arredondar
df_model_metrics = pd.DataFrame(metrics).round(2)
df_model_metrics 

,whisper_version,model,accuracy_label,precision_label,recall_label,f1_label,accuracy_categoria,precision_categoria,recall_categoria,f1_categoria
0,large-v3_results.pkl,deepseek-r1-8b,0.33,0.72,0.33,0.43,0.35,0.41,0.35,0.30
1,large_results.pkl,deepseek-r1-8b,0.35,0.79,0.35,0.47,0.33,0.41,0.33,0.31
2,tiny_results.pkl,deepseek-r1-8b,0.17,0.59,0.17,0.26,0.33,0.31,0.33,0.30
3,large-v3_results.pkl,gemma2-9b,0.22,0.91,0.22,0.35,0.38,0.31,0.38,0.32
4,large_results.pkl,gemma2-9b,0.23,0.96,0.23,0.37,0.38,0.31,0.38,0.31
5,tiny_results.pkl,gemma2-9b,0.15,0.88,0.15,0.24,0.31,0.25,0.31,0.26
6,large-v3_results.pkl,gpt-4.1-mini,0.34,0.79,0.34,0.47,0.49,0.57,0.49,0.47
7,large_results.pkl,gpt-4.1-mini,0.32,0.75,0.32,0.45,0.46,0.53,0.46,0.43
8,tiny_results.pkl,gpt-4.1-mini,0.25,0.91,0.25,0.38,0.42,0.45,0.42,0.38
9,large-v3_results.pkl,gpt-4o-mini,0.07,1.00,0.07,0.13,0.40,0.47,0.40,0.34


In [36]:
melhor_whisper = df_summary['overall_score'].idxmax()
print("Melhor whisper_version:", melhor_whisper)
print(df_summary.loc[melhor_whisper])

Melhor whisper_version: large-v3_results.pkl
f1_label              0.3930
f1_categoria          0.3400
accuracy_label        0.3290
accuracy_categoria    0.3920
mean_f1               0.3665
mean_accuracy         0.3605
overall_score         0.3635
Name: large-v3_results.pkl, dtype: float64


In [95]:
# Adiciona um score médio por linha
df_model_metrics['mean_f1'] = df_model_metrics[['f1_label', 'f1_categoria']].mean(axis=1)

# Seleciona top-N modelos por média de f1
top_models = (
    df_model_metrics
    .groupby('model')['mean_f1']
    .mean()
    .sort_values(ascending=False)
    .head(4)
    .index.tolist()
)
top_models

['llama3.1-8b', 'gpt-4.1-mini', 'phi3-14b', 'llama3.2-3b']

In [139]:
from sklearn.metrics import f1_score

# Filtra amostras válidas para ambas as tasks
df_filtrado = df_predict[df_predict['label_real'] != 2]
df_filtrado = df_filtrado[~df_filtrado['video_id'].isin(['2', '127'])]  # Se desejar manter o mesmo filtro anterior

# Armazenar resultados
f1_label_scores = []
f1_categoria_scores = []

# Agrupar por modelo
for model_name, group in df_filtrado.groupby('model'):
    # Tarefa: label
    f1_label = f1_score(group['label_real'], group['label'], average='weighted', zero_division=0)
    f1_label_scores.append((model_name, f1_label))

    # Tarefa: categoria
    f1_categoria = f1_score(group['categoria_real'], group['categoria'], average='weighted', zero_division=0)
    f1_categoria_scores.append((model_name, f1_categoria))

# Ordenar os top-4 modelos por F1-score (maior para menor)
top5_label = sorted(f1_label_scores, key=lambda x: x[1], reverse=True)[:6]
top5_categoria = sorted(f1_categoria_scores, key=lambda x: x[1], reverse=True)[:6]

# Exibir resultados
print("Top 4 models for label (F1 weighted):")
for model, score in top5_label:
    print(f"{model}: {score:.3f}")

print("\nTop 4 models for categoria (F1 weighted):")
for model, score in top5_categoria:
    print(f"{model}: {score:.3f}")


Top 4 models for label (F1 weighted):
llama3.1-8b: 0.650
llama3.2-3b: 0.548
gpt-4.1-mini: 0.436
deepseek-r1-8b: 0.392
phi3-14b: 0.389
qwen2.5-14b: 0.368

Top 4 models for categoria (F1 weighted):
gpt-4.1-mini: 0.429
phi3-14b: 0.418
qwen2.5-14b: 0.329
gpt-4o-mini: 0.327
qwen2-7b: 0.322
qwen2.5-7b: 0.313


In [140]:
top4_label

[('llama3.1-8b', 0.6495908496207141),
 ('llama3.2-3b', 0.5482169027845628),
 ('gpt-4.1-mini', 0.4361086129367001),
 ('deepseek-r1-8b', 0.39177139037433156),
 ('phi3-14b', 0.38891366677285766)]

In [142]:
# Filtrar df para os modelos top de label
df_label_top = df_filtrado[df_filtrado['model'].isin(top_models_label)]
df_categoria_top = df_filtrado[df_filtrado['model'].isin(top_models_categoria)]

# Função para calcular F1-score por whisper_version
def get_f1_by_whisper(df, y_true_col, y_pred_col):
    return (
        df.groupby('whisper_version')
        .apply(lambda g: f1_score(g[y_true_col], g[y_pred_col], average='weighted', zero_division=0))
        .sort_values(ascending=False)
    )


In [143]:
f1_by_whisper_label = get_f1_by_whisper(df_label_top, 'label_real', 'label')
f1_by_whisper_categoria = get_f1_by_whisper(df_categoria_top, 'categoria_real', 'categoria')

print("F1 por whisper (label):")
print(f1_by_whisper_label)

print("\nF1 por whisper (categoria):")
print(f1_by_whisper_categoria)


F1 por whisper (label):
whisper_version
large_results.pkl       0.550306
large-v3_results.pkl    0.534447
tiny_results.pkl        0.450048
dtype: float64

F1 por whisper (categoria):
whisper_version
large-v3_results.pkl    0.375365
large_results.pkl       0.364017
tiny_results.pkl        0.352711
dtype: float64


C:\Users\usuario\AppData\Local\Temp\ipykernel_13708\4256242595.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: f1_score(g[y_true_col], g[y_pred_col], average='weighted', zero_division=0))
C:\Users\usuario\AppData\Local\Temp\ipykernel_13708\4256242595.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: f1_score(g[y_true_col], g[y_pred_col], average='weighted', zero_divis

In [144]:
from scipy.stats import wilcoxon

# Preparar comparação entre whispers nos top-models
def compare_whispers(df, y_true_col, y_pred_col, melhor='large-v3_results.pkl'):
    pivot = (
        df.groupby(['model', 'whisper_version'])
        .apply(lambda g: f1_score(g[y_true_col], g[y_pred_col], average='weighted', zero_division=0))
        .unstack()
    )

    comparacoes = {}
    for col in pivot.columns:
        if col == melhor:
            continue
        try:
            stat, pval = wilcoxon(pivot[melhor], pivot[col])
            comparacoes[col] = round(pval, 4)
        except ValueError:
            comparacoes[col] = np.nan
    return comparacoes
print("\nWilcoxon — task: LABEL")
pvals_label = compare_whispers(df_label_top, 'label_real', 'label')
print(pvals_label)

print("\nWilcoxon — task: CATEGORIA")
pvals_cat = compare_whispers(df_categoria_top, 'categoria_real', 'categoria')
print(pvals_cat)


Wilcoxon — task: LABEL
{'large_results.pkl': np.float64(0.3125), 'tiny_results.pkl': np.float64(0.0625)}

Wilcoxon — task: CATEGORIA
{'large_results.pkl': np.float64(0.4375), 'tiny_results.pkl': np.float64(0.3125)}


C:\Users\usuario\AppData\Local\Temp\ipykernel_13708\379651033.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: f1_score(g[y_true_col], g[y_pred_col], average='weighted', zero_division=0))
C:\Users\usuario\AppData\Local\Temp\ipykernel_13708\379651033.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: f1_score(g[y_true_col], g[y_pred_col], average='weighted', zero_divisio

In [147]:
df_label_top.columns

Index(['video_id', 'model', 'whisper_version', 'label', 'label_real',
       'categoria', 'categoria_real'],
      dtype='object')

In [148]:
import pandas as pd
from sklearn.metrics import f1_score
from scipy.stats import friedmanchisquare

# Filtra amostras válidas
df_filtrado = df_label_top[df_label_top['label_real'] != 2]
df_filtrado = df_filtrado[~df_filtrado['video_id'].isin(['2', '127'])]

# Passo 1 — calcular F1s por (model, whisper_version)
metrics = []

for (model, whisper), group in df_filtrado.groupby(['model', 'whisper_version']):
    f1_label = f1_score(group['label_real'], group['label'], average='weighted', zero_division=0)
    f1_cat = f1_score(group['categoria_real'], group['categoria'], average='weighted', zero_division=0)
    mean_f1 = (f1_label + f1_cat) / 2

    metrics.append({
        'model': model,
        'whisper_version': whisper,
        'f1_label': f1_label,
        'f1_categoria': f1_cat,
        'mean_f1': mean_f1
    })

df_metrics = pd.DataFrame(metrics)

# Passo 2 — Pivot: cada linha é um modelo, cada coluna é um whisper_version
pivot = df_metrics.pivot(index='model', columns='whisper_version', values='mean_f1')

# Passo 3 — Aplicar Friedman
stat, p_value = friedmanchisquare(*[pivot[col] for col in pivot.columns])

# Resultado
print(f"Friedman stat = {stat:.4f}, p-value = {p_value:.4f}")

Friedman stat = 9.3333, p-value = 0.0094


In [149]:
import pandas as pd
from sklearn.metrics import f1_score
from scipy.stats import friedmanchisquare

# Filtra amostras válidas
df_filtrado = df_categoria_top[df_categoria_top['categoria_real'] != 2]
df_filtrado = df_filtrado[~df_filtrado['video_id'].isin(['2', '127'])]

# Passo 1 — calcular F1s por (model, whisper_version)
metrics = []

for (model, whisper), group in df_filtrado.groupby(['model', 'whisper_version']):
    f1_categoria = f1_score(group['categoria_real'], group['categoria'], average='weighted', zero_division=0)
    f1_cat = f1_score(group['categoria_real'], group['categoria'], average='weighted', zero_division=0)
    mean_f1 = (f1_categoria + f1_cat) / 2

    metrics.append({
        'model': model,
        'whisper_version': whisper,
        'f1_categoria': f1_categoria,
        'f1_categoria': f1_cat,
        'mean_f1': mean_f1
    })

df_metrics = pd.DataFrame(metrics)

# Passo 2 — Pivot: cada linha é um modelo, cada coluna é um whisper_version
pivot = df_metrics.pivot(index='model', columns='whisper_version', values='mean_f1')

# Passo 3 — Aplicar Friedman
stat, p_value = friedmanchisquare(*[pivot[col] for col in pivot.columns])

# Resultado
print(f"Friedman stat = {stat:.4f}, p-value = {p_value:.4f}")

Friedman stat = 0.3333, p-value = 0.8465


## Multimodal Prompet

In [24]:
mapa_parecer = {
    'veridico': 1,
    'verdadeiro': 1,
    'inveridico': 0,
    'inconclusivo': 2
}

mapa_categoria = {
    'descontextualizacao': 0,
    'fake news': 1,
    'nao se aplica': 2,
    'distorcao de midia': -1,
    'extremismo antidemocratico': 4,
    'discurso de odio': 5
}

In [ ]:
import pandas as pd
import unicodedata

result = pd.read_pickle(r"result\final_result_curadoria.pkl")

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('utf-8')
    return x

def limpar_id(x):
    x = str(x).strip()
    if x.endswith('.mp4'):
        x = x[:-4]
    try:
        return str(int(float(x))) 
    except:
        return x

def normalizar(texto):
    texto = texto.strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto

# Aplicar normalização nas colunas 'parecer' e 'categoria'
result['parecer'] = result['parecer'].apply(normalizar_texto)
result['categoria'] = result['categoria'].apply(normalizar_texto)

result.rename(columns={'id':'video_id'}, inplace=True)
result['video_id'] = result['video_id'].apply(limpar_id)

result['categoria'] = result['categoria'].apply(lambda x: mapa_categoria.get(normalizar(x), 2))
result['parecer'] = result['parecer'].apply(lambda x: mapa_parecer.get(normalizar(x), 2))
df_real = result.copy()


import os

# Caminho principal
base_path = r'results_llm_video_description_v3'

# Lista para armazenar os DataFrames
dataframes = []

# Percorre todas as subpastas
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith('.pkl'):
            file_path = os.path.join(root, file)
            model = os.path.basename(root)
            whisper_version = file

            # Lê o arquivo pkl
            try:
                df = pd.read_pickle(file_path)
                df['model'] = model
                df['whisper_version'] = whisper_version
                dataframes.append(df)
            except Exception as e:
                print(f"Erro ao ler {file_path}: {e}")

# Concatena todos os DataFrames
df_total_raw = pd.concat(dataframes, ignore_index=True)
df_total_raw = df_total_raw.loc[:, ~df_total_raw.columns.str.startswith('embedding_')]
df_total_raw = df_total_raw.loc[:, ~df_total_raw.columns.str.startswith('nome_base')]


df_total_raw = df_total_raw.dropna(subset=['label', 'categoria'])
df_total_raw['video_id'] = pd.to_numeric(df_total_raw['video_id'], errors='coerce').astype('Int64')
df_total_raw = df_total_raw.dropna(subset=['video_id'])
df_total_raw['video_id'] = df_total_raw['video_id'].astype(int)

df_total = df_total_raw.copy()

df_total['label'] = df_total['label'].apply(normalizar_texto)
df_total['categoria'] = df_total['categoria'].apply(normalizar_texto)

df_total['video_id'] = df_total['video_id'].apply(limpar_id)

df_total['categoria'] = df_total['categoria'].apply(lambda x: mapa_categoria.get(normalizar(x), 2))
df_total['label'] = df_total['label'].apply(lambda x: mapa_parecer.get(normalizar(x), 2))

# Função para limpar IDs (tira espaços, converte para int se possível e depois para str)
def limpar_id(x):
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()

# Aplicar nos dois DataFrames
df_total['video_id'] = df_total['video_id'].apply(limpar_id)
df_real['video_id'] = df_real['video_id'].apply(limpar_id)

# Selecionar e renomear as colunas
result_filtrado = df_real[['video_id', 'categoria', 'parecer']].rename(
    columns={
        'categoria': 'categoria_real',
        'parecer': 'label_real'
    }
)

# Garantir tipos numéricos (int)
result_filtrado['categoria_real'] = result_filtrado['categoria_real'].astype(int)
result_filtrado['label_real'] = result_filtrado['label_real'].astype(int)

# Fazer o merge
df_total = df_total.merge(result_filtrado, on='video_id', how='left')
df_predict = df_total[['video_id', 'model', 'whisper_version', 'label', 'label_real', 'categoria', 'categoria_real']]

#Ajustando labels não normalizadas corretamente

df_predict.loc[(df_predict['video_id'] == '172') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0
df_predict.loc[(df_predict['video_id'] == '153') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0
df_predict.loc[(df_predict['video_id'] == '141') & (df_predict['model'] == 'phi3-14b'), 'label'] = 1
df_predict.loc[(df_predict['video_id'] == '134') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0
df_predict.loc[(df_predict['video_id'] == '1') & (df_predict['model'] == 'phi3-14b'), 'label'] = 0

df_predict

,video_id,model,whisper_version,label,label_real,categoria,categoria_real
0,28,deepseek-r1-32b,large-v3_results.pkl,1,2,2,2
1,14,deepseek-r1-32b,large-v3_results.pkl,0,0,4,1
2,33,deepseek-r1-32b,large-v3_results.pkl,0,0,5,4
3,113,deepseek-r1-32b,large-v3_results.pkl,0,0,4,0
4,40,deepseek-r1-32b,large-v3_results.pkl,0,2,0,0
...,...,...,...,...,...,...,...
1423,198,qwen2.5-7b,large-v3_results.pkl,0,0,2,5
1424,199,qwen2.5-7b,large-v3_results.pkl,0,1,2,2
1425,200,qwen2.5-7b,large-v3_results.pkl,0,1,0,2
1426,202,qwen2.5-7b,large-v3_results.pkl,0,0,4,5


In [26]:
from sklearn.metrics import classification_report, f1_score
import pandas as pd

relatorios = []

# Filtrar valores válidos
df_filtrado = df_predict.dropna(subset=['label', 'label_real'])
df_filtrado = df_filtrado[
    (df_filtrado['label_real'] != 2) &
    (~df_filtrado['video_id'].isin(['2', '127']))
]

# Agrupar por modelo e whisper_version
for model, grupo in df_filtrado.groupby('model'):
    y_true = grupo['label_real']
    y_pred = grupo['label']

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    resumo = {
        'model': model
    }

    # Adiciona métricas por label (0 e 1)
    for label in ['0', '1']:
        if label in report:
            # resumo[f'precision_label_{label}'] = round(report[label]['precision'], 3)
            # resumo[f'recall_label_{label}'] = round(report[label]['recall'], 3)
            resumo[f'f1_label_{label}'] = round(report[label]['f1-score'], 3)
        else:
            # resumo[f'precision_label_{label}'] = 0.0
            # resumo[f'recall_label_{label}'] = 0.0
            resumo[f'f1_label_{label}'] = 0.0

    # Adiciona F1 macro total
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    resumo['f1_label_macro'] = round(f1_macro, 3)

    relatorios.append(resumo)

# Criação do DataFrame final
df_metricas_por_label = pd.DataFrame(relatorios)
df_metricas_por_label = df_metricas_por_label.sort_values(by='f1_label_macro', ascending=False)
print(df_metricas_por_label)

              model  f1_label_0  f1_label_1  f1_label_macro
2         gemma2-9b       0.889       0.638           0.764
4      gpt-4.1-mini       0.815       0.615           0.715
6       llama3.1-8b       0.838       0.538           0.688
9          qwen2-7b       0.679       0.615           0.647
0   deepseek-r1-32b       0.831       0.435           0.633
1    deepseek-r1-8b       0.704       0.507           0.605
3        gemma3-12b       0.838       0.350           0.594
10      qwen2.5-14b       0.838       0.350           0.594
5       gpt-4o-mini       0.841       0.278           0.560
8          phi3-14b       0.549       0.531           0.540
11       qwen2.5-7b       0.831       0.176           0.504
7       llama3.2-3b       0.817       0.000           0.408


In [27]:
from sklearn.metrics import classification_report, f1_score
import pandas as pd

relatorios = []

# Filtrar valores válidos
df_filtrado = df_predict.dropna(subset=['categoria', 'categoria_real'])
# df_filtrado = df_filtrado[~df_filtrado['video_id'].isin(['2', '127'])]

# Agrupar por modelo
for model, grupo in df_filtrado.groupby('model'):
    y_true = grupo['categoria_real']
    y_pred = grupo['categoria']

    # Gera o relatório completo
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    resumo = {'model': model}

    # Adiciona f1-score por classe individual
    # for categoria in sorted(df_filtrado['categoria_real'].unique()):
    #     cat_str = str(categoria)
    #     if cat_str in report:
    #         resumo[f'f1_categoria_{cat_str}'] = round(report[cat_str]['f1-score'], 3)
    #     else:
    #         resumo[f'f1_categoria_{cat_str}'] = 0.0

    # F1 macro total
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    resumo['f1_categoria_macro'] = round(f1_macro, 3)

    relatorios.append(resumo)

# Criar DataFrame e ordenar pelo f1_macro
df_metricas_por_categoria = pd.DataFrame(relatorios)
df_metricas_por_categoria = df_metricas_por_categoria.sort_values(by='f1_categoria_macro', ascending=False)

# Exibir resultado
print(df_metricas_por_categoria)

              model  f1_categoria_macro
3        gemma3-12b               0.524
4      gpt-4.1-mini               0.485
5       gpt-4o-mini               0.475
10      qwen2.5-14b               0.442
2         gemma2-9b               0.344
0   deepseek-r1-32b               0.326
7       llama3.2-3b               0.290
11       qwen2.5-7b               0.270
1    deepseek-r1-8b               0.252
6       llama3.1-8b               0.247
9          qwen2-7b               0.246
8          phi3-14b               0.137


In [31]:
from sklearn.metrics import classification_report
import pandas as pd

relatorios = []

# Filtra valores válidos para garantir que não haja erro na comparação
df_filtrado = df_predict.dropna(subset=['categoria', 'categoria_real'])

# Agrupa por modelo e versão do whisper
grupos = df_filtrado.groupby(['model', 'whisper_version'])

for (model, version), grupo in grupos:
    y_true = grupo['categoria_real']
    y_pred = grupo['categoria']

    # Gera o relatório
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    # Extrai as métricas desejadas
    resumo = {
        'model': model,
        'accuracy': round(report['accuracy'], 2),
        'precision_macro': round(report['macro avg']['precision'], 2),
        'recall_macro': round(report['macro avg']['recall'], 2),
        'f1_macro': round(report['macro avg']['f1-score'], 2),
        'precision_weighted': round(report['weighted avg']['precision'], 2),
        'recall_weighted': round(report['weighted avg']['recall'], 2),
        'f1_weighted': round(report['weighted avg']['f1-score'], 2)
    }

    relatorios.append(resumo)

# Converte em DataFrame
df_metricas_categoria = pd.DataFrame(relatorios)
# Calcula a média de cada métrica numérica
media_geral = df_metricas_categoria.drop(columns=['model']).mean().round(3)

# Cria uma nova linha com os valores médios e identificadores
linha_media = {
    'model': 'MÉDIA GERAL',
    **media_geral.to_dict()
}

# Adiciona ao DataFrame
df_metricas_categoria_com_media = pd.concat([
    df_metricas_categoria,
    pd.DataFrame([linha_media])
], ignore_index=True)
df_metricas_categoria_com_media

,model,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,deepseek-r1-32b,0.410,0.420,0.370,0.330,0.450,0.410,0.350
1,deepseek-r1-8b,0.310,0.320,0.260,0.250,0.340,0.310,0.280
2,gemma2-9b,0.390,0.630,0.370,0.340,0.640,0.390,0.350
3,gemma3-12b,0.550,0.580,0.540,0.520,0.600,0.550,0.540
4,gpt-4.1-mini,0.520,0.560,0.470,0.480,0.540,0.520,0.500
5,gpt-4o-mini,0.520,0.580,0.490,0.470,0.580,0.520,0.500
6,llama3.1-8b,0.340,0.300,0.280,0.250,0.340,0.340,0.280
7,llama3.2-3b,0.320,0.300,0.290,0.290,0.340,0.320,0.320
8,phi3-14b,0.230,0.340,0.240,0.140,0.460,0.230,0.160
9,qwen2-7b,0.330,0.270,0.260,0.250,0.330,0.330,0.300


In [32]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Garante pasta de saída
outdir = "confusion_matrices_categoria"
os.makedirs(outdir, exist_ok=True)

# Usa o mesmo filtro do seu código
df_filtrado = df_predict.dropna(subset=['categoria', 'categoria_real'])

# Ordem global de rótulos (consistente entre modelos)
labels = sorted(
    set(df_filtrado['categoria_real'].unique()).union(
        set(df_filtrado['categoria'].unique())
    )
)

# Agrupa por modelo e versão do whisper
grupos = df_filtrado.groupby(['model', 'whisper_version'])

# Escolha a normalização: None, 'true' (por linha/verdadeiro) ou 'pred' (por coluna/predito)
normalize_mode = 'true'  # mude para None se quiser contagens absolutas

for (model, version), grupo in grupos:
    y_true = grupo['categoria_real'].to_numpy()
    y_pred = grupo['categoria'].to_numpy()

    # Matriz de confusão
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize=normalize_mode)

    # Plot
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    disp.plot(ax=ax, cmap='Blues', values_format='.2f' if normalize_mode else 'd', colorbar=False)
    ax.set_title(f'Confusion Matrix — {model} | {version}\n(normalize={normalize_mode})', pad=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Nome seguro de arquivo
    safe = lambda s: re.sub(r'[^A-Za-z0-9_.-]+', '_', str(s))
    fname = f"{safe(model)}__{safe(version)}.png"
    plt.savefig(os.path.join(outdir, fname), dpi=200)
    plt.close(fig)

print(f"Matrizes salvas em: ./{outdir}")


Matrizes salvas em: ./confusion_matrices_categoria


In [30]:
from sklearn.metrics import classification_report
import pandas as pd

relatorios = []

# Filtra valores válidos para garantir que não haja erro na comparação
df_filtrado = df_predict.dropna(subset=['label', 'label_real'])

# Agrupa por modelo e versão do whisper
grupos = df_filtrado.groupby(['model', 'whisper_version'])

for (model, version), grupo in grupos:
    y_true = grupo['label_real']
    y_pred = grupo['label']

    # Gera o relatório
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    # Extrai as métricas desejadas
    resumo = {
        'model': model,
        'accuracy': round(report['accuracy'], 2),
        # 'precision_macro': round(report['macro avg']['precision'], 2),
        # 'recall_macro': round(report['macro avg']['recall'], 2),
        # 'f1_macro': round(report['macro avg']['f1-score'], 2),
        # 'precision_weighted': round(report['weighted avg']['precision'], 2),
        # 'recall_weighted': round(report['weighted avg']['recall'], 2),
        # 'f1_weighted': round(report['weighted avg']['f1-score'], 2)
    }

    relatorios.append(resumo)

# Converte em DataFrame
df_metricas_label = pd.DataFrame(relatorios)
# Calcula a média de cada métrica numérica
media_geral = df_metricas_label.drop(columns=['model']).mean().round(3)

# Cria uma nova linha com os valores médios e identificadores
linha_media = {
    'model': 'MÉDIA GERAL',
    **media_geral.to_dict()
}

# Adiciona ao DataFrame
df_metricas_label_com_media = pd.concat([
    df_metricas_label,
    pd.DataFrame([linha_media])
], ignore_index=True)
print(df_metricas_label_com_media)

              model  accuracy
0   deepseek-r1-32b     0.630
1    deepseek-r1-8b     0.540
2         gemma2-9b     0.710
3        gemma3-12b     0.640
4      gpt-4.1-mini     0.640
5       gpt-4o-mini     0.630
6       llama3.1-8b     0.650
7       llama3.2-3b     0.590
8          phi3-14b     0.460
9          qwen2-7b     0.550
10      qwen2.5-14b     0.630
11       qwen2.5-7b     0.610
12      MÉDIA GERAL     0.607
